# Laptop Price Predictor — Major Project

Predict laptop price from **Brand, RAM, Processor Family and Storage** using Linear Regression.

Dataset: CampusX public GitHub CSV. Evaluation uses held-out MAE, RMSE and R².

In [ ]:
!pip -q install pandas numpy scikit-learn joblib flask


In [ ]:
import re, joblib, numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
DATA_URL = 'https://raw.githubusercontent.com/campusx-official/laptop-price-predictor-regression-project/main/laptop_data.csv'
df = pd.read_csv(DATA_URL)
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
def parse_ram(value):
    m = re.search(r'(\d+(?:\.\d+)?)', str(value))
    return float(m.group(1)) if m else np.nan

def parse_storage(value):
    total = 0.0
    for amount, unit in re.findall(r'(\d+(?:\.\d+)?)\s*(TB|GB)', str(value).upper().replace(',', '')):
        total += float(amount) * (1024 if unit == 'TB' else 1)
    return total if total else np.nan

def processor_family(value):
    cpu = str(value).lower()
    for token, label in [('core i9','Intel Core i9'),('core i7','Intel Core i7'),('core i5','Intel Core i5'),('core i3','Intel Core i3'),('core m','Intel Core M'),('celeron','Intel Celeron'),('pentium','Intel Pentium'),('atom','Intel Atom')]:
        if token in cpu: return label
    if 'amd' in cpu: return 'AMD'
    if 'intel' in cpu: return 'Intel Other'
    return 'Other'

work = pd.DataFrame({'Brand':df['Company'].astype('string').fillna('Unknown').str.strip(), 'RAM_GB':df['Ram'].map(parse_ram), 'Processor_Family':df['Cpu'].map(processor_family), 'Storage_GB':df['Memory'].map(parse_storage), 'Price':pd.to_numeric(df['Price'],errors='coerce')}).dropna(subset=['Price'])
X = work[['Brand','RAM_GB','Processor_Family','Storage_GB']]
y = work['Price']
print('Prepared rows:', len(work))
display(work.head())


In [ ]:
categorical = ['Brand','Processor_Family']
numeric = ['RAM_GB','Storage_GB']
preprocessor = ColumnTransformer([
 ('categorical', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), categorical),
 ('numeric', Pipeline([('imputer',SimpleImputer(strategy='median'))]), numeric)
])
model = Pipeline([('preprocessor',preprocessor),('regressor',LinearRegression())])
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))
r2 = r2_score(y_test,y_pred)
print(f'MAE: {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R2: {r2:.4f}')


In [ ]:
bundle = {'model':model,'feature_columns':X.columns.tolist(),'brand_options':sorted(X.Brand.unique().tolist()),'processor_options':sorted(X.Processor_Family.unique().tolist()),'metrics':{'MAE':float(mae),'RMSE':float(rmse),'R2':float(r2),'train_rows':len(X_train),'test_rows':len(X_test)}}
joblib.dump(bundle,'laptop_price_model.joblib')
print('Saved laptop_price_model.joblib')


## Example prediction
The same four input fields used by the Flask form are passed to the fitted pipeline.

In [ ]:
sample = pd.DataFrame([{'Brand':X.Brand.mode()[0], 'RAM_GB':8, 'Processor_Family':X.Processor_Family.mode()[0], 'Storage_GB':512}])
print('Example predicted price:', round(float(model.predict(sample)[0]),2))
